# Assumption: the data from all the treated reservoirs is available up to the same date

### Import modules and data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import sys

# Setting up the path to include the parent directory
sys.path.append(str(Path.cwd().parent.parent))
from backend.config.settings import PATHS
from backend.transferring.transfer_optimization import make_optimal_transfers

critical_threshold = 0.075
worrying_threshold = 0.15
could_give_if_critical = 0.25
could_give_if_worrying = 0.35
able_to_donate = 0.6
cost_threshold = 1.5
epsilon = 1e-6

In [ ]:
water_path = PATHS['definitive_notebooks'] / 'water_definitive.parquet'
water_pd = pd.read_parquet(water_path)
water_pd.head()

In [ ]:
reservoirs_path = PATHS['definitive_notebooks'] / 'reservoirs_merged.parquet'
reservoirs_pd = pd.read_parquet(reservoirs_path)
reservoirs_pd.head()

In [ ]:
def plot_all_reservoirs(reservoirs_list, with_transfers=False, legend=True):
    id_to_capacity = reservoirs_pd.set_index('id')['capacity'].to_dict()
    difference = {}
    transfers_log = pd.DataFrame()
    if with_transfers:
        
        _, transfers_log = make_optimal_transfers(reservoirs_list)

        for _, row in transfers_log.iterrows():
            if row['donor'] not in difference:
                difference[row['donor']] = 0
            difference[row['donor']] -= row['difference']
            if row['receiver'] not in difference:
                difference[row['receiver']] = 0
            difference[row['receiver']] += row['difference']

    next_year_predictions_path = PATHS['processed_data_notebooks'] / 'next_year_predictions.parquet'
    next_year_predictions = pd.read_parquet(next_year_predictions_path)

    water_pd['capacity'] = water_pd['id'].map(id_to_capacity)

    plt.figure(figsize=(12, 6))

    for reservoir in reservoirs_list:
        reservoir_pd = water_pd[water_pd['id'] == reservoir]
        reservoir_pd.set_index('date', inplace=True)
        df1 = reservoir_pd['storage'][-52*4:]/id_to_capacity[reservoir]
        df2 = (next_year_predictions[reservoir] + difference.get(reservoir, 0))/id_to_capacity[reservoir]
        df = pd.concat([df1, df2], axis=0)
        
        plt.plot(df.index,
                df,
                label=f'Reservoir {reservoir}')
        
    plt.axvline(x=df1.index[-1], color='k', linestyle='--', label='Next Year Prediction')
    plt.xlabel('Date')
    plt.ylabel('Storage (%)')
    plt.title('Evolution of Reservoir Storage')
    if legend:
        plt.legend()
    plt.show()
    return transfers_log

In [ ]:
reservoirs_list = reservoirs_pd[reservoirs_pd['province']=='huesca']['id'].values
reservoirs_list

In [ ]:
transfers_log = plot_all_reservoirs(reservoirs_list, with_transfers=False, legend=False)

In [ ]:
transfers_log = plot_all_reservoirs(reservoirs_list, with_transfers=True, legend=False)

In [ ]:
def plot_all_reservoirs_involved(reservoirs_list, with_transfers=False, legend=True):
    id_to_capacity = reservoirs_pd.set_index('id')['capacity'].to_dict()
    difference = {}
    _, transfers_log = make_optimal_transfers(reservoirs_list)

    for _, row in transfers_log.iterrows():
            if row['donor'] not in difference:
                difference[row['donor']] = 0
            difference[row['donor']] -= row['difference']
            if row['receiver'] not in difference:
                difference[row['receiver']] = 0
            difference[row['receiver']] += row['difference']
        
    reservoirs_involved = difference.keys()
    if not with_transfers:
         difference = {}

    next_year_predictions_path = PATHS['processed_data_notebooks'] / 'next_year_predictions.parquet'
    next_year_predictions = pd.read_parquet(next_year_predictions_path)

    water_pd['capacity'] = water_pd['id'].map(id_to_capacity)

    plt.figure(figsize=(12, 6))

    for reservoir in reservoirs_involved:
        reservoir_pd = water_pd[water_pd['id'] == reservoir]
        reservoir_pd.set_index('date', inplace=True)
        df1 = reservoir_pd['storage'][-52*4:]/id_to_capacity[reservoir]
        df2 = (next_year_predictions[reservoir] + difference.get(reservoir, 0))/id_to_capacity[reservoir]
        df = pd.concat([df1, df2], axis=0)
        
        plt.plot(df.index,
                df,
                label=f'Reservoir {int(reservoir)}')
        
    plt.axvline(x=df1.index[-1], color='k', linestyle='--', label='Next Year Prediction')
    plt.xlabel('Date')
    plt.ylabel('Storage (%)')
    plt.title('Evolution of Reservoir Storage')
    if legend:
        plt.legend()
    plt.show()
    return transfers_log

In [ ]:
plot_all_reservoirs_involved(reservoirs_list, with_transfers=False, legend=True)

In [ ]:
plot_all_reservoirs_involved(reservoirs_list, with_transfers=True, legend=True)